In [1]:
import pandas as pd
import numpy as np
import optuna

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, balanced_accuracy_score
from xgboost import XGBClassifier

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X = train.drop(columns=["id", "Irrigation_Need"])
y = train["Irrigation_Need"]

X_test = test.drop(columns=["id"])
test_ids = test["id"]

# one-hot encode for RF and XGBoost
X = pd.get_dummies(X, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# align columns
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [5]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scorer = make_scorer(balanced_accuracy_score)

In [6]:
sample_size = 100000 
train_sample = train.sample(sample_size, random_state=42)

X_sample = train_sample.drop(columns=["id", "Irrigation_Need"])
y_sample = train_sample["Irrigation_Need"]

X_sample = pd.get_dummies(X_sample, drop_first=True)
X_sample = X_sample.reindex(columns=X.columns, fill_value=0)

y_sample_encoded = le.transform(y_sample)


def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 400),
        "max_depth": trial.suggest_int("max_depth", 8, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 42,
        "n_jobs": -1
    }

    model = RandomForestClassifier(**params)

    score = cross_val_score(
        model,
        X_sample,
        y_sample_encoded,
        cv=3,
        scoring="balanced_accuracy",
        n_jobs=-1
    ).mean()

    return score


study_rf = optuna.create_study(direction="maximize")

study_rf.optimize(objective_rf, n_trials=5)

print("Best RF score:", study_rf.best_value)
print("Best RF params:", study_rf.best_params)

[I 2026-04-15 23:01:03,897] A new study created in memory with name: no-name-741bfbc0-bc80-4cc8-b1e6-bdc1d40b5c24
[I 2026-04-15 23:01:27,368] Trial 0 finished with value: 0.9405891134000567 and parameters: {'n_estimators': 284, 'max_depth': 19, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9405891134000567.
[I 2026-04-15 23:01:51,094] Trial 1 finished with value: 0.9327387582516091 and parameters: {'n_estimators': 379, 'max_depth': 20, 'min_samples_split': 6, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9405891134000567.
[I 2026-04-15 23:02:08,362] Trial 2 finished with value: 0.7937143021932046 and parameters: {'n_estimators': 382, 'max_depth': 10, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.9405891134000567.
[I 2026-04-15 23:02:19,077] Trial 3 finished with value: 0.7225109824584651 and parameters: {'n_estimators': 273, 'max_depth': 8, 'mi

Best RF score: 0.9405891134000567
Best RF params: {'n_estimators': 284, 'max_depth': 19, 'min_samples_split': 8, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


In [7]:
best_rf = RandomForestClassifier(
    **study_rf.best_params,
    random_state=42,
    n_jobs=-1
)

best_rf.fit(X, y_encoded)

rf_preds = best_rf.predict(X_test)
rf_preds = le.inverse_transform(rf_preds)

bagging_submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": rf_preds
})

bagging_submission.to_csv("bagging_submission.csv", index=False)

In [9]:
sample_size = 100000 
train_sample = train.sample(sample_size, random_state=42)

X_sample = train_sample.drop(columns=["id", "Irrigation_Need"])
y_sample = train_sample["Irrigation_Need"]

X_sample = pd.get_dummies(X_sample, drop_first=True)
X_sample = X_sample.reindex(columns=X.columns, fill_value=0)

y_sample_encoded = le.transform(y_sample)


def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15, log=True),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 6),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "objective": "multi:softprob",
        "num_class": 3,
        "random_state": 42,
        "eval_metric": "mlogloss",
        "n_jobs": -1,
        "tree_method": "hist"
    }

    model = XGBClassifier(**params)

    score = cross_val_score(
        model,
        X_sample,  
        y_sample_encoded,
        cv=3,        
        scoring="balanced_accuracy",
        n_jobs=1                 
    ).mean()

    return score


study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=5)

print("Best XGB score:", study_xgb.best_value)
print("Best XGB params:", study_xgb.best_params)

[I 2026-04-15 23:06:51,733] A new study created in memory with name: no-name-e46a2a4e-2e4e-4f2d-a1a6-173593948e51
[I 2026-04-15 23:06:58,154] Trial 0 finished with value: 0.9579977732863026 and parameters: {'n_estimators': 383, 'max_depth': 8, 'learning_rate': 0.10217480540538633, 'subsample': 0.8124858609551449, 'colsample_bytree': 0.9626385116201832, 'min_child_weight': 4, 'gamma': 1.1386635509624519}. Best is trial 0 with value: 0.9579977732863026.
[I 2026-04-15 23:07:08,153] Trial 1 finished with value: 0.9571724074376503 and parameters: {'n_estimators': 459, 'max_depth': 5, 'learning_rate': 0.041193559267010266, 'subsample': 0.8256643708391298, 'colsample_bytree': 0.9629882136050658, 'min_child_weight': 5, 'gamma': 0.2606452226750835}. Best is trial 0 with value: 0.9579977732863026.
[I 2026-04-15 23:07:13,105] Trial 2 finished with value: 0.9570235503765949 and parameters: {'n_estimators': 321, 'max_depth': 5, 'learning_rate': 0.14370306768843832, 'subsample': 0.7451960527685078, 

Best XGB score: 0.9579977732863026
Best XGB params: {'n_estimators': 383, 'max_depth': 8, 'learning_rate': 0.10217480540538633, 'subsample': 0.8124858609551449, 'colsample_bytree': 0.9626385116201832, 'min_child_weight': 4, 'gamma': 1.1386635509624519}


In [10]:
best_xgb = XGBClassifier(
    **study_xgb.best_params,
    objective="multi:softmax",
    num_class=3,
    random_state=42,
    eval_metric="mlogloss",
    n_jobs=-1
)

best_xgb.fit(X, y_encoded)

xgb_preds = best_xgb.predict(X_test)
xgb_preds = le.inverse_transform(xgb_preds)

boosting_submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": xgb_preds
})

boosting_submission.to_csv("boosting_submission.csv", index=False)